# C-GASTON CONCH — Stats Test + Boundary Precision
**Sharvani Sinha — EECS 545**

Computes everything needed to fill:
- **Backbone comparison sheet**: Moran's I for UNI and CONCH (std)
- **Stats_Significance sheet**: Mann-Whitney U p-values + Boundary Precision@50µm mean±std

Uses saved `.npy` files from training — no retraining needed.

## Step 0: Install

In [1]:
import subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('anndata', 'scanpy', 'scipy', 'scikit-learn', 'libpysal', 'esda')
pip('gaston-spatial')
print('Done.')

Done.


## Step 1: Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('Mounted.')

Mounted at /content/drive
Mounted.


## Step 2: Imports

In [3]:
import os
import numpy as np
import scipy.sparse as sp
import anndata as ad
from scipy.stats import mannwhitneyu
from scipy.spatial import cKDTree
from libpysal.weights import KNN
from esda.moran import Moran
import warnings
warnings.filterwarnings('ignore')
print('Imports done.')

Imports done.


## Step 3: Configuration
**Edit the paths below if your Drive folders are named differently.**

In [4]:
# ── PATHS ─────────────────────────────────────────────────────
BASE_DIR        = '/content/drive/MyDrive/EECS 545 Project'
GLMPCA_DIR      = '/content/drive/MyDrive/EECS545_glmpca_results'
HE_BASE_DIR     = f'{BASE_DIR}/DLPFC_Datasets(raw data)'
SAMPLE1_DIR     = f'{HE_BASE_DIR}/Sample1/h5ad_cordintae_data'
SAMPLE3_DIR     = f'{HE_BASE_DIR}/Sample3/h5ad_cordinate_data'

# Saved isodepth + label .npy files from each training run
CONCH_STD_DIR   = '/content/drive/MyDrive/C_GASTON_results_conch'
CONCH_SOFT_DIR  = '/content/drive/MyDrive/C_GASTON_results_conch_soft'
UNI_STD_DIR     = '/content/drive/MyDrive/C_GASTON_results/standard'
# GASTON baseline — adjust if your path differs
GASTON_DIR      = '/content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8'
# ─────────────────────────────────────────────────────────────

ALL_SLICES = ['151507','151508','151509','151510',
              '151673','151674','151675','151676']

SLICE_DATA_DIRS = {
    '151507': SAMPLE1_DIR, '151508': SAMPLE1_DIR,
    '151509': SAMPLE1_DIR, '151510': SAMPLE1_DIR,
    '151673': SAMPLE3_DIR, '151674': SAMPLE3_DIR,
    '151675': SAMPLE3_DIR, '151676': SAMPLE3_DIR,
}
LABEL_TO_INT    = {'L1':0,'L2':1,'L3':2,'L4':3,'L5':4,'L6':5,'WM':6}
INTER_SPOT_UM   = 100.0   # µm between Visium spot centres
THRESHOLD_UM    = 50.0    # Boundary Precision threshold

print('Config loaded.')

Config loaded.


## Step 4: Load spatial coords + ground truth for all slices
Needed for Moran's I and Boundary Precision.

In [5]:
data = {}
for sid in ALL_SLICES:
    adata = ad.read_h5ad(f'{SLICE_DATA_DIRS[sid]}/{sid}.h5ad')
    adata.var_names_make_unique()
    S      = np.asarray(adata.obsm['spatial'])
    gt_str = adata.obs['original_domain'].astype(str).values
    gt_int = np.array([LABEL_TO_INT.get(l, -1) for l in gt_str])
    data[sid] = {'coords': S, 'gt_labels': gt_int}
    print(f'{sid}: {S.shape[0]} spots')

print('\n[OK] Coords + GT loaded.')

151507: 4221 spots
151508: 4381 spots
151509: 4788 spots
151510: 4595 spots
151673: 3611 spots
151674: 3635 spots
151675: 3566 spots
151676: 3431 spots

[OK] Coords + GT loaded.


## Step 5: Helper functions

In [6]:
# ── Moran's I ─────────────────────────────────────────────────
def compute_morans_i(coords, isodepth, k=6):
    w = KNN.from_array(coords, k=k)
    w.transform = 'r'
    return Moran(isodepth, w).I

def morans_all_slices(iso_dir, iso_file):
    vals = []
    for sid in ALL_SLICES:
        path = f'{iso_dir}/{sid}/{iso_file}'
        if not os.path.isfile(path):
            print(f'  {sid}: FILE NOT FOUND — {path}')
            vals.append(np.nan)
            continue
        iso = np.load(path)
        mi  = compute_morans_i(data[sid]['coords'].astype(float), iso)
        vals.append(mi)
        print(f'  {sid}: {mi:.4f}')
    mean = np.nanmean(vals)
    print(f'  MEAN: {mean:.4f}')
    return vals, mean


# ── Boundary Precision@50µm ────────────────────────────────────
def get_boundary_mask(coords_um, labels, neighbor_mult=1.5):
    unique_x   = np.unique(coords_um[:, 0])
    px_spacing = np.median(np.diff(np.sort(unique_x)))
    radius     = px_spacing * neighbor_mult
    tree       = cKDTree(coords_um)
    mask       = np.zeros(len(coords_um), dtype=bool)
    for i, (c, lbl) in enumerate(zip(coords_um, labels)):
        nbrs = [j for j in tree.query_ball_point(c, r=radius) if j != i]
        if any(labels[j] != lbl for j in nbrs):
            mask[i] = True
    return mask

def boundary_precision_score(coords, gt_labels, pred_labels):
    """Convert pixel coords to µm, then compute BP@50µm."""
    unique_x   = np.unique(coords[:, 0])
    px_spacing = np.median(np.diff(np.sort(unique_x)))
    coords_um  = coords * (INTER_SPOT_UM / px_spacing)

    valid = gt_labels >= 0
    c, gt, pr = coords_um[valid], gt_labels[valid], pred_labels[valid]

    gt_bnd   = get_boundary_mask(c, gt)
    pred_bnd = get_boundary_mask(c, pr)
    if gt_bnd.sum() == 0 or pred_bnd.sum() == 0:
        return 0.0

    tree    = cKDTree(c[gt_bnd])
    dists,_ = tree.query(c[pred_bnd], k=1)
    return float((dists <= THRESHOLD_UM).mean())

def boundary_precision_all_slices(label_dir, label_file):
    bps = []
    for sid in ALL_SLICES:
        path = f'{label_dir}/{sid}/{label_file}'
        if not os.path.isfile(path):
            print(f'  {sid}: FILE NOT FOUND — {path}')
            bps.append(np.nan)
            continue
        pred   = np.load(path).astype(int)
        coords = data[sid]['coords'].astype(float)
        gt     = data[sid]['gt_labels']
        bp     = boundary_precision_score(coords, gt, pred)
        bps.append(bp)
        print(f'  {sid}: {bp:.4f}')
    mean = np.nanmean(bps)
    std  = np.nanstd(bps)
    print(f'  Mean={mean:.4f}  Std={std:.4f}')
    return bps, mean, std

print('Helpers defined.')

Helpers defined.


## Step 6: Moran's I


In [11]:
print("=" * 50)
print("Moran's I — UNI (Standard)")
print("=" * 50)
uni_mi_vals, uni_mi_mean = morans_all_slices(UNI_STD_DIR, 'cgaston_isodepth.npy')

print()
print("=" * 50)
print("Moran's I — CONCH (Standard)")
print("=" * 50)
conch_mi_vals, conch_mi_mean = morans_all_slices(CONCH_STD_DIR, 'cgaston_isodepth.npy')

print()
print("=" * 50)
print("SUMMARY ")
print("=" * 50)
print(f"  UNI   Moran's I = {uni_mi_mean:.4f}")
print(f"  CONCH Moran's I = {conch_mi_mean:.4f}")
print(f"  ResNet-50 (already in sheet) = 0.9940")
print(f"  UNI   Δ Moran's I = {uni_mi_mean - 0.9940:+.4f}")
print(f"  CONCH Δ Moran's I = {conch_mi_mean - 0.9940:+.4f}")

Moran's I — UNI (Standard)
  151507: 0.9967
  151508: 0.9792
  151509: 0.9972
  151510: 0.9975
  151673: 0.9961
  151674: 0.9924
  151675: 0.9882
  151676: 0.9967
  MEAN: 0.9930

Moran's I — CONCH (Standard)
  151507: 0.9966
  151508: 0.9770
  151509: 0.9971
  151510: 0.9975
  151673: 0.9961
  151674: 0.9930
  151675: 0.9960
  151676: 0.9967
  MEAN: 0.9938

SUMMARY 
  UNI   Moran's I = 0.9930
  CONCH Moran's I = 0.9938
  ResNet-50 (already in sheet) = 0.9940
  UNI   Δ Moran's I = -0.0010
  CONCH Δ Moran's I = -0.0002


## Step 7: Mann-Whitney U p-values


In [8]:
# From Main sheet (already filled — GASTON baseline)
gaston_aris = [0.3998, 0.4174, 0.3891, 0.3448,
               0.5575, 0.5112, 0.4342, 0.5567]

# From C-GASTON std column in Main sheet
cgaston_std_aris  = [0.4819, 0.4076, 0.3884, 0.3775,
                     0.7003, 0.4997, 0.4949, 0.5451]

# From C-GASTON soft column in Main sheet
cgaston_soft_aris = [0.4796, 0.4216, 0.3971, 0.2860,
                     0.6266, 0.4806, 0.4184, 0.5293]

_, p1 = mannwhitneyu(cgaston_std_aris,  gaston_aris,       alternative='two-sided')
_, p2 = mannwhitneyu(cgaston_soft_aris, gaston_aris,       alternative='two-sided')
_, p3 = mannwhitneyu(cgaston_std_aris,  cgaston_soft_aris, alternative='two-sided')

print("=" * 55)
print("Mann-Whitney U — enter in Stats_Significance sheet")
print("=" * 55)
print(f"  Row 2 | C-GASTON Std  vs GASTON (ARI):        p = {p1:.4f}")
print(f"  Row 3 | C-GASTON Soft vs GASTON (ARI):        p = {p2:.4f}")
print(f"  Row 4 | C-GASTON Std  vs C-GASTON Soft (ARI): p = {p3:.4f}")
print()
if p1 < 0.05:
    print("  ✓ Std vs GASTON is significant (p < 0.05)")
else:
    print("  ✗ Std vs GASTON is NOT significant (p ≥ 0.05)")
if p2 < 0.05:
    print("  ✓ Soft vs GASTON is significant (p < 0.05)")
else:
    print("  ✗ Soft vs GASTON is NOT significant (p ≥ 0.05)")

Mann-Whitney U — enter in Stats_Significance sheet
  Row 2 | C-GASTON Std  vs GASTON (ARI):        p = 0.8785
  Row 3 | C-GASTON Soft vs GASTON (ARI):        p = 0.8785
  Row 4 | C-GASTON Std  vs C-GASTON Soft (ARI): p = 0.6454

  ✗ Std vs GASTON is NOT significant (p ≥ 0.05)
  ✗ Soft vs GASTON is NOT significant (p ≥ 0.05)


## Step 8: Boundary Precision@50µm

Computes for GASTON baseline, CONCH Std, and CONCH Soft.

In [12]:
print("=" * 50)
print("Boundary Precision@50µm — GASTON")
print("=" * 50)
bp_gaston, bp_gaston_mean, bp_gaston_std = boundary_precision_all_slices(
    GASTON_DIR, 'gaston_labels.npy'
)

print()
print("=" * 50)
print("Boundary Precision@50µm — CONCH Standard")
print("=" * 50)
bp_std, bp_std_mean, bp_std_std = boundary_precision_all_slices(
    CONCH_STD_DIR, 'cgaston_labels.npy'
)

print()
print("=" * 50)
print("Boundary Precision@50µm — CONCH Soft")
print("=" * 50)
bp_soft, bp_soft_mean, bp_soft_std = boundary_precision_all_slices(
    CONCH_SOFT_DIR, 'cgaston_soft_labels.npy'
)

print()
print("=" * 55)
print("SUMMARY")
print("=" * 55)
print(f"  Row 9  | GASTON:        {bp_gaston_mean:.4f} ± {bp_gaston_std:.4f}")
print(f"  Row 10 | C-GASTON Std:  {bp_std_mean:.4f} ± {bp_std_std:.4f}")
print(f"  Row 11 | C-GASTON Soft: {bp_soft_mean:.4f} ± {bp_soft_std:.4f}")

Boundary Precision@50µm — GASTON
  151507: FILE NOT FOUND — /content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8/151507/gaston_labels.npy
  151508: FILE NOT FOUND — /content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8/151508/gaston_labels.npy
  151509: FILE NOT FOUND — /content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8/151509/gaston_labels.npy
  151510: FILE NOT FOUND — /content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8/151510/gaston_labels.npy
  151673: FILE NOT FOUND — /content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8/151673/gaston_labels.npy
  151674: FILE NOT FOUND — /content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8/151674/gaston_labels.npy
  151675: FILE NOT FOUND — /content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8/151675/gaston_labels.npy
  151676: FILE NOT FOUND — /content/drive/MyDrive/Evaluation_Comparison/gaston_baseline_all8/151676/gaston_labels.npy
  Mean=nan  Std=nan

Bo

## Step 9: Full printout for the Excel sheet
All numbers in one place.

In [14]:


print("\n── Backbone comparison sheet ──")
print(f"  UNI row:")
print(f"    ARI      = 0.4691  (mean of 8 slices from UNI notebook)")
print(f"    NMI      = 0.6147")
print(f"    Spearman = 0.6977")
print(f"    Moran's I = {uni_mi_mean:.4f}")
print(f"  CONCH row:")
print(f"    ARI      = 0.4792  (mean of 8 slices from CONCH notebook)")
print(f"    NMI      = 0.6234")
print(f"    Spearman = 0.7657")
print(f"    Moran's I = {conch_mi_mean:.4f}")
print(f"  Δ UNI − ResNet-50:")
print(f"    ΔARI={0.4691-0.4869:+.4f}  ΔNMI={0.6147-0.6247:+.4f}  ΔSpearman={0.6977-0.7229:+.4f}  ΔMoran={uni_mi_mean-0.994:+.4f}")
print(f"  Δ CONCH − ResNet-50:")
print(f"    ΔARI={0.4792-0.4869:+.4f}  ΔNMI={0.6234-0.6247:+.4f}  ΔSpearman={0.7657-0.7229:+.4f}  ΔMoran={conch_mi_mean-0.994:+.4f}")

print("\n── Stats_Significance sheet ──")
print(f"  Row 2 | C-GASTON Std  vs GASTON:        p = {p1:.4f}")
print(f"  Row 3 | C-GASTON Soft vs GASTON:        p = {p2:.4f}")
print(f"  Row 4 | C-GASTON Std  vs Soft:          p = {p3:.4f}")
print(f"  Row 9  | GASTON:        {bp_gaston_mean:.4f} ± {bp_gaston_std:.4f}")
print(f"  Row 10 | C-GASTON Std:  {bp_std_mean:.4f} ± {bp_std_std:.4f}")
print(f"  Row 11 | C-GASTON Soft: {bp_soft_mean:.4f} ± {bp_soft_std:.4f}")
print("=" * 60)


── Backbone comparison sheet ──
  UNI row:
    ARI      = 0.4691  (mean of 8 slices from UNI notebook)
    NMI      = 0.6147
    Spearman = 0.6977
    Moran's I = 0.9930
  CONCH row:
    ARI      = 0.4792  (mean of 8 slices from CONCH notebook)
    NMI      = 0.6234
    Spearman = 0.7657
    Moran's I = 0.9938
  Δ UNI − ResNet-50:
    ΔARI=-0.0178  ΔNMI=-0.0100  ΔSpearman=-0.0252  ΔMoran=-0.0010
  Δ CONCH − ResNet-50:
    ΔARI=-0.0077  ΔNMI=-0.0013  ΔSpearman=+0.0428  ΔMoran=-0.0002

── Stats_Significance sheet ──
  Row 2 | C-GASTON Std  vs GASTON:        p = 0.8785
  Row 3 | C-GASTON Soft vs GASTON:        p = 0.8785
  Row 4 | C-GASTON Std  vs Soft:          p = 0.6454
  Row 9  | GASTON:        nan ± nan
  Row 10 | C-GASTON Std:  0.3746 ± 0.0902
  Row 11 | C-GASTON Soft: 0.3650 ± 0.0670
